# 🏥 CT DICOM Visualization — Kaggle CT Medical Images

Reproduces and extends the analysis from  
**[Visualize CT DICOM Data](https://www.kaggle.com/code/gpreda/visualize-ct-dicom-data/notebook)** by Gabriel Preda

**Dataset:** [CT Medical Images](https://www.kaggle.com/datasets/kmader/siim-medical-images) — `kmader/siim-medical-images`  
475 CT series from 69 patients, sourced from the Cancer Imaging Archive (TCIA).

---

## 📥 Step 0 — Download the Dataset

### Option A — Kaggle CLI (recommended)
```bash
pip install kaggle
# Place your kaggle.json API token in ~/.kaggle/kaggle.json
kaggle datasets download -d kmader/siim-medical-images
unzip siim-medical-images.zip -d siim-medical-images/
```

### Option B — Inside a Kaggle Notebook
The data is already mounted at:
```
/kaggle/input/siim-medical-images/
```

### Dataset folder layout
```
siim-medical-images/
├── dicom_dir/          ← DICOM files (.dcm), one per patient series
├── full_archive.csv    ← metadata: PatientID, Age, Contrast, Modality…
└── overview.csv        ← summary stats
```

In [ ]:
# ── Install dependencies ───────────────────────────────────────────────────────
# !pip install pydicom matplotlib numpy pandas scipy pillow seaborn

import os
import glob
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy.ndimage import sobel
from PIL import Image
import io

try:
    import pydicom
    from pydicom import dcmread
    PYDICOM_OK = True
    print(f'✓ pydicom {pydicom.__version__}')
except ImportError:
    PYDICOM_OK = False
    print('✗ pydicom not installed — pip install pydicom')

print(f'✓ numpy {np.__version__}')
import matplotlib; print(f'✓ matplotlib {matplotlib.__version__}')

---
## 1️⃣  Configure Paths

In [ ]:
# ── Set ONE of these to match your environment ────────────────────────────────

# Local download
DATA_DIR = './siim-medical-images'

# Kaggle Notebook
# DATA_DIR = '/kaggle/input/siim-medical-images'

# ─────────────────────────────────────────────────────────────────────────────
DICOM_DIR = os.path.join(DATA_DIR, 'dicom_dir')
CSV_PATH  = os.path.join(DATA_DIR, 'full_archive.csv')

# ── Verify ────────────────────────────────────────────────────────────────────
DATA_AVAILABLE = os.path.isdir(DICOM_DIR)
CSV_AVAILABLE  = os.path.isfile(CSV_PATH)

if DATA_AVAILABLE:
    dcm_files = sorted(glob.glob(os.path.join(DICOM_DIR, '**', '*.dcm'), recursive=True))
    # Also accept flat layout
    if not dcm_files:
        dcm_files = sorted(glob.glob(os.path.join(DICOM_DIR, '*.dcm')))
    print(f'✓ Found {len(dcm_files)} DICOM files in {DICOM_DIR}')
else:
    print(f'✗ DICOM directory not found at: {DICOM_DIR}')
    print('  Running in DEMO mode with synthetic data.')
    dcm_files = []

if CSV_AVAILABLE:
    print(f'✓ Metadata CSV found: {CSV_PATH}')
else:
    print(f'  (CSV not found — metadata section will be skipped)')

---
## 2️⃣  DICOM vs JPEG/PNG — Key Differences

| Feature | DICOM (`.dcm`) | JPEG / PNG |
|---|---|---|
| **Purpose** | Medical imaging standard (ISO 12052) | General-purpose display format |
| **Bit Depth** | **12–16 bit** (up to 65 536 gray levels) | 8 bit per channel (256 levels) |
| **Metadata** | Full clinical record: patient ID, modality, FOV, kVp… | Minimal EXIF only |
| **Compression** | Lossless or uncompressed | JPEG = lossy; PNG = lossless |
| **Multi-frame** | Yes — entire CT/MRI stack in one file | No — one file per image |
| **Pixel values** | Hounsfield Units (absolute tissue density) | sRGB display values |
| **PHI** | Embeds Protected Health Information | No PHI |

> **Why JPEG is forbidden in clinical DICOM pipelines:**  
> A CT scan spans ~4 000 unique HU values. Mapping to 8-bit discards **94 %** of the diagnostic range,  
> potentially hiding low-contrast lesions that differ from healthy tissue by only 5–10 HU.

---
## 3️⃣  Load the Metadata CSV

In [ ]:
def make_synthetic_metadata(n=60):
    """Synthetic dataframe matching the shape of full_archive.csv."""
    rng = np.random.default_rng(0)
    ages = rng.integers(25, 85, n)
    return pd.DataFrame({
        'PatientID' : [f'ID-{i:04d}' for i in range(n)],
        'Age'       : ages,
        'Contrast'  : rng.choice([0, 1], n),
        'Modality'  : rng.choice(['CT', 'CTA'], n, p=[0.6, 0.4]),
    })


if CSV_AVAILABLE:
    df = pd.read_csv(CSV_PATH)
    print(f'Shape: {df.shape}')
    print(df.head())
else:
    df = make_synthetic_metadata()
    print('Using synthetic metadata (CSV not found).')
    print(df.head())

print('\nColumns:', list(df.columns))

---
## 4️⃣  Exploratory Data Analysis on Metadata

In [ ]:
BG = '#0d1117'
PANEL = '#161b22'
BORDER = '#30363d'
BLUE = '#38bdf8'
GREEN = '#34d399'
PINK = '#fb7185'
YELLOW = '#fbbf24'
LABEL = '#94a3b8'
TITLE = '#7dd3fc'

# Normalise column names for robustness
df.columns = [c.strip() for c in df.columns]
age_col   = next((c for c in df.columns if 'age' in c.lower()), None)
mod_col   = next((c for c in df.columns if 'modal' in c.lower()), None)
con_col   = next((c for c in df.columns if 'contrast' in c.lower()), None)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.patch.set_facecolor(BG)
fig.suptitle('CT Medical Images — Dataset Overview', color='white',
             fontsize=14, fontweight='bold')

def style_ax(ax, title):
    ax.set_facecolor(PANEL)
    ax.set_title(title, color=TITLE, fontsize=11)
    ax.tick_params(colors=LABEL)
    ax.set_xlabel('', color=LABEL)
    for s in ax.spines.values(): s.set_edgecolor(BORDER)

# ── Age distribution ─────────────────────────────────────────────────────────
ax = axes[0]
if age_col:
    ages = pd.to_numeric(df[age_col], errors='coerce').dropna()
    ax.hist(ages, bins=20, color=BLUE, edgecolor=BG)
    ax.set_xlabel('Age', color=LABEL)
    ax.set_ylabel('Count', color=LABEL)
else:
    ax.text(0.5, 0.5, 'Age column\nnot found', ha='center', va='center',
            color=LABEL, transform=ax.transAxes)
style_ax(ax, 'Patient Age Distribution')

# ── Modality breakdown ───────────────────────────────────────────────────────
ax = axes[1]
if mod_col:
    counts = df[mod_col].value_counts()
    bars = ax.bar(counts.index, counts.values,
                  color=[GREEN, YELLOW, BLUE, PINK][:len(counts)], edgecolor=BG)
    ax.set_ylabel('Count', color=LABEL)
    for bar, val in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, val + 0.5, str(val),
                ha='center', va='bottom', color='white', fontsize=9)
else:
    ax.text(0.5, 0.5, 'Modality column\nnot found', ha='center', va='center',
            color=LABEL, transform=ax.transAxes)
style_ax(ax, 'Modality (CT vs CTA)')

# ── Contrast breakdown ───────────────────────────────────────────────────────
ax = axes[2]
if con_col:
    counts = df[con_col].value_counts()
    labels = [f'Contrast {k}' for k in counts.index]
    ax.pie(counts.values, labels=labels, autopct='%1.0f%%',
           colors=[PINK, BLUE], textprops={'color': 'white'},
           wedgeprops={'edgecolor': BG, 'linewidth': 2})
else:
    ax.text(0.5, 0.5, 'Contrast column\nnot found', ha='center', va='center',
            color=LABEL, transform=ax.transAxes)
style_ax(ax, 'Contrast Usage')

plt.tight_layout()
plt.show()

---
## 5️⃣  Load a Single DICOM File & Inspect Tags

In [ ]:
def synthetic_dicom():
    """Realistic synthetic CT head slice for demo mode."""
    rng = np.random.default_rng(7)
    size = 512
    img = np.full((size, size), -1000, dtype=np.int16)
    yy, xx = np.ogrid[:size, :size]
    brain = ((xx-256)**2/180**2 + (yy-256)**2/220**2) < 1
    skull = ((xx-256)**2/200**2 + (yy-256)**2/240**2) < 1
    img[skull]  = (700 + rng.integers(-40, 40, (size,size)))[skull]
    img[brain]  = ( 40 + rng.integers(-25, 25, (size,size)))[brain]
    img[230:250, 275:295] = 75   # small bright region

    class FakeDCM:
        PatientID = 'DEMO-001'; PatientName = 'Demo^Patient'
        PatientAge = '052Y'; PatientSex = 'F'
        StudyDate = '20230819'; Modality = 'CT'
        StudyDescription = 'CT ABDOMEN/PELVIS'
        SliceThickness = 2.5; PixelSpacing = [0.703, 0.703]
        RescaleSlope = 1.0; RescaleIntercept = -1024.0
        BitsAllocated = 16; Rows = size; Columns = size
        pixel_array = img
        def __repr__(self): return f'SyntheticDICOM({self.Rows}×{self.Columns})'
    return FakeDCM()


def load_dcm(path):
    if not PYDICOM_OK:
        return synthetic_dicom()
    return dcmread(path)


# ── Pick the first available file, or fall back to synthetic ─────────────────
if dcm_files:
    ds = load_dcm(dcm_files[0])
    print(f'Loaded: {dcm_files[0]}')
else:
    ds = synthetic_dicom()
    print('Using synthetic DICOM (no files found).')

# ── Print key tags ────────────────────────────────────────────────────────────
tags = ['PatientID','PatientName','PatientAge','PatientSex',
        'StudyDate','Modality','StudyDescription',
        'Rows','Columns','BitsAllocated',
        'SliceThickness','PixelSpacing','RescaleSlope','RescaleIntercept']

print('\n{:<28} {}'.format('TAG', 'VALUE'))
print('─'*55)
for t in tags:
    val = getattr(ds, t, 'N/A')
    print(f'{t:<28} {val}')

---
## 6️⃣  Hounsfield Unit Conversion

```
HU = raw_pixel × RescaleSlope + RescaleIntercept
```

| Tissue | HU |
|---|---|
| Air | −1000 |
| Lung | −600 → −400 |
| Fat | −100 → −50 |
| Water / CSF | 0 |
| Brain | 20 → 80 |
| Blood | 30 → 45 |
| Bone | 400 → 1000 |

In [ ]:
def to_hu(ds):
    px = ds.pixel_array.astype(np.float32)
    s  = float(getattr(ds, 'RescaleSlope',     1.0))
    i  = float(getattr(ds, 'RescaleIntercept', 0.0))
    return px * s + i


def apply_window(hu, wc, ww):
    lo, hi = wc - ww/2, wc + ww/2
    return ((np.clip(hu, lo, hi) - lo) / (hi - lo) * 255).astype(np.uint8)


hu = to_hu(ds)
print(f'Shape   : {hu.shape}')
print(f'HU range: [{hu.min():.0f}, {hu.max():.0f}]')
print(f'Mean HU : {hu.mean():.1f}   Std: {hu.std():.1f}')
print(f'Unique  : {len(np.unique(hu))} values  '
      f'(vs 256 max in 8-bit JPEG/PNG)')

---
## 7️⃣  Clinical Windowing

In [ ]:
WINDOWS = {
    'Brain'    : ( 40,   80),
    'Subdural' : ( 75,  215),
    'Stroke'   : ( 32,   48),
    'Bone'     : (400, 1800),
    'Lung'     : (-600,1500),
    'Abdomen'  : ( 60,  400),
}

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
fig.patch.set_facecolor(BG)
fig.suptitle('CT Windowing — Same DICOM slice, six clinical presets',
             color='white', fontsize=14, fontweight='bold', y=1.01)

for ax, (name, (wc, ww)) in zip(axes.flat, WINDOWS.items()):
    ax.set_facecolor(PANEL)
    ax.imshow(apply_window(hu, wc, ww), cmap='gray', vmin=0, vmax=255)
    ax.set_title(f'{name}\nWC={wc}  WW={ww}', color=TITLE, fontsize=10)
    ax.axis('off')

plt.tight_layout()
plt.show()

---
## 8️⃣  Histogram: DICOM 16-bit vs 8-bit JPEG

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor(BG)
fig.suptitle('Pixel Value Distribution — DICOM (16-bit HU) vs Simulated JPEG (8-bit)',
             color='white', fontsize=13, fontweight='bold')

HU_ANNOTATIONS = [
    (-1000, 'Air',   '#94a3b8'),
    (  -50, 'Fat',   '#a78bfa'),
    (    0, 'Water', '#60a5fa'),
    (   40, 'Brain', '#f9a8d4'),
    (  700, 'Bone',  '#fbbf24'),
]

# Left: full HU histogram
ax0 = axes[0]
ax0.set_facecolor(PANEL)
flat = hu.flatten()
ax0.hist(flat, bins=200, color=BLUE, edgecolor='none', alpha=0.85)
ymax = ax0.get_ylim()[1]
for hv, lab, col in HU_ANNOTATIONS:
    if flat.min() <= hv <= flat.max():
        ax0.axvline(hv, color=col, lw=1.2, linestyle='--')
        ax0.text(hv + 30, ymax * 0.6, lab, color=col, fontsize=8, rotation=90, va='top')
ax0.set_title(f'DICOM — {len(np.unique(flat))} unique HU values', color=TITLE)
ax0.set_xlabel('HU Value', color=LABEL); ax0.set_ylabel('Count', color=LABEL)
ax0.tick_params(colors=LABEL)
for s in ax0.spines.values(): s.set_edgecolor(BORDER)

# Right: 8-bit brain-windowed histogram
ax1 = axes[1]
ax1.set_facecolor(PANEL)
bw = apply_window(hu, 40, 80)
ax1.hist(bw.flatten(), bins=64, color=PINK, edgecolor='none', alpha=0.85)
ax1.set_title('Simulated 8-bit (Brain Window)\n94% of HU range discarded', color=TITLE)
ax1.set_xlabel('Value (0–255)', color=LABEL); ax1.set_ylabel('Count', color=LABEL)
ax1.tick_params(colors=LABEL)
for s in ax1.spines.values(): s.set_edgecolor(BORDER)

# Summary boxes
for ax, txt in [
    (ax0, f'Unique values\n{len(np.unique(flat))}\n(16-bit DICOM)'),
    (ax1, f'Unique values\n{len(np.unique(bw))}\n(8-bit JPEG/PNG)'),
]:
    ax.text(0.97, 0.97, txt, transform=ax.transAxes, fontsize=9,
            va='top', ha='right', color='white',
            bbox=dict(boxstyle='round,pad=0.4', fc='#21262d', ec=BORDER))

plt.tight_layout()
plt.show()

---
## 9️⃣  Full Analysis Dashboard

In [ ]:
fig = plt.figure(figsize=(18, 11))
fig.patch.set_facecolor(BG)
gs = gridspec.GridSpec(2, 4, figure=fig, hspace=0.42, wspace=0.35)

brain_w  = apply_window(hu, 40, 80)
bone_w   = apply_window(hu, 400, 1800)
lung_w   = apply_window(hu, -600, 1500)

# Row 0 — four image panels
for spec, img, cmap, title in [
    (gs[0,0], brain_w,  'gray',     'Brain Window (WC=40, WW=80)'),
    (gs[0,1], brain_w,  'hot',      'Brain — Hot colormap'),
    (gs[0,2], bone_w,   'bone',     'Bone Window (WC=400, WW=1800)'),
    (gs[0,3], lung_w,   'gist_gray','Lung Window (WC=−600, WW=1500)'),
]:
    ax = fig.add_subplot(spec)
    ax.set_facecolor(PANEL)
    im = ax.imshow(img, cmap=cmap)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04).ax.tick_params(colors=LABEL, labelsize=7)
    ax.set_title(title, color=TITLE, fontsize=9, pad=5)
    ax.axis('off')

# Row 1a — edge map
ax_e = fig.add_subplot(gs[1,0])
ax_e.set_facecolor(PANEL)
sx = sobel(brain_w.astype(float), axis=0)
sy = sobel(brain_w.astype(float), axis=1)
ax_e.imshow(np.hypot(sx, sy), cmap='inferno')
ax_e.set_title('Sobel Edge Map', color=TITLE, fontsize=9)
ax_e.axis('off')

# Row 1b — HU intensity profile
ax_p = fig.add_subplot(gs[1,1])
ax_p.set_facecolor(PANEL)
mid = hu[hu.shape[0]//2, :]
ax_p.plot(mid, color=BLUE, lw=1.2)
for hv, lab, col in [(-1000,'Air','#94a3b8'),(0,'Water','#60a5fa'),(400,'Bone',YELLOW)]:
    ax_p.axhline(hv, color=col, lw=0.8, ls='--', label=f'{lab} ({hv} HU)')
ax_p.set_title('HU Profile — Centre Row', color=TITLE, fontsize=9)
ax_p.set_xlabel('Column', color=LABEL, fontsize=7); ax_p.set_ylabel('HU', color=LABEL, fontsize=7)
ax_p.tick_params(colors=LABEL, labelsize=7)
ax_p.legend(fontsize=7, labelcolor='white', facecolor='#21262d', edgecolor=BORDER)
for s in ax_p.spines.values(): s.set_edgecolor(BORDER)

# Row 1c — tissue segmentation
ax_s = fig.add_subplot(gs[1,2])
ax_s.set_facecolor(PANEL)
seg = np.zeros((*hu.shape, 3), dtype=np.uint8)
seg[hu < -500]                         = [20,  20,  20]
seg[(hu>=-500) & (hu<-50)]             = [90, 160, 230]
seg[(hu>= -50) & (hu< 200)]            = [240,160,140]
seg[hu >= 200]                         = [250,215,100]
ax_s.imshow(seg)
ax_s.legend(handles=[
    mpatches.Patch(color=np.array([20,20,20])/255,   label='Air'),
    mpatches.Patch(color=np.array([90,160,230])/255, label='Lung/Fat'),
    mpatches.Patch(color=np.array([240,160,140])/255,label='Soft Tissue'),
    mpatches.Patch(color=np.array([250,215,100])/255,label='Bone'),
], loc='lower right', fontsize=7, labelcolor='white', facecolor='#21262d', edgecolor=BORDER)
ax_s.set_title('Tissue Segmentation (HU Thresholds)', color=TITLE, fontsize=9)
ax_s.axis('off')

# Row 1d — stats
ax_i = fig.add_subplot(gs[1,3])
ax_i.set_facecolor(PANEL); ax_i.axis('off')
stats = (
    f"DICOM IMAGE STATS\n{'─'*24}\n"
    f"Modality    {getattr(ds,'Modality','CT')}\n"
    f"Patient ID  {getattr(ds,'PatientID','N/A')}\n"
    f"Shape       {hu.shape[0]} × {hu.shape[1]}\n"
    f"Bits alloc  {getattr(ds,'BitsAllocated',16)}\n"
    f"Px spacing  {getattr(ds,'PixelSpacing',['?','?'])[0]} mm\n"
    f"Slice thick {getattr(ds,'SliceThickness','?')} mm\n"
    f"{'─'*24}\n"
    f"HU min      {hu.min():.0f}\n"
    f"HU max      {hu.max():.0f}\n"
    f"HU mean     {hu.mean():.1f}\n"
    f"HU std      {hu.std():.1f}\n"
    f"Unique vals {len(np.unique(hu))}"
)
ax_i.text(0.05, 0.97, stats, transform=ax_i.transAxes, fontsize=9,
          va='top', color='#e2e8f0', family='monospace',
          bbox=dict(boxstyle='round,pad=0.6', fc='#21262d', ec=BORDER))
ax_i.set_title('Image Info', color=TITLE, fontsize=9)

fig.suptitle('DICOM Analysis Dashboard — CT Medical Images (Kaggle kmader/siim-medical-images)',
             color='white', fontsize=13, fontweight='bold', y=1.01)
plt.show()

---
## 🔟  Browse Multiple Files from the Dataset

In [ ]:
def load_series(file_list, n=12, wc=40, ww=400):
    """Load up to n DICOM files and return a list of windowed arrays."""
    images, labels = [], []
    for path in file_list[:n]:
        try:
            d = load_dcm(path)
            h = to_hu(d)
            images.append(apply_window(h, wc, ww))
            labels.append(os.path.basename(path)[:20])
        except Exception as e:
            print(f'Skip {path}: {e}')
    return images, labels


if dcm_files:
    imgs, lbls = load_series(dcm_files, n=12)
else:
    # Simulate 12 CT slices synthetically
    rng = np.random.default_rng(99)
    imgs, lbls = [], []
    for i in range(12):
        scale = 1 - abs(i - 5) / 8
        imgs.append(apply_window(hu * scale + rng.normal(0, 8, hu.shape), 40, 400))
        lbls.append(f'slice_{i+1:03d}.dcm')

cols = 4
rows = int(np.ceil(len(imgs) / cols))
fig, axes = plt.subplots(rows, cols, figsize=(cols*4, rows*4))
fig.patch.set_facecolor(BG)
fig.suptitle(f'CT Medical Images — {len(imgs)} Samples (Abdomen Window WC=40, WW=400)',
             color='white', fontsize=13, fontweight='bold')

for i, ax in enumerate(axes.flat):
    ax.set_facecolor(PANEL)
    if i < len(imgs):
        ax.imshow(imgs[i], cmap='gray')
        ax.set_title(lbls[i], color=TITLE, fontsize=8)
    ax.axis('off')

plt.tight_layout()
plt.show()

---
## 1️⃣1️⃣  Scatter: Patient Age vs Mean HU (from CSV)

In [ ]:
def compute_mean_hu_for_files(file_list, max_files=60):
    rows = []
    for path in file_list[:max_files]:
        try:
            d = load_dcm(path)
            h = to_hu(d)
            rows.append({
                'file'     : os.path.basename(path),
                'mean_hu'  : float(h.mean()),
                'std_hu'   : float(h.std()),
                'modality' : getattr(d, 'Modality', 'CT'),
            })
        except Exception:
            pass
    return pd.DataFrame(rows)


if dcm_files:
    stats_df = compute_mean_hu_for_files(dcm_files, max_files=60)
else:
    rng2 = np.random.default_rng(42)
    stats_df = pd.DataFrame({
        'file'     : [f'img_{i}.dcm' for i in range(60)],
        'mean_hu'  : rng2.normal(-300, 200, 60),
        'std_hu'   : rng2.uniform(200, 600, 60),
        'modality' : rng2.choice(['CT','CTA'], 60),
    })

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor(BG)
fig.suptitle('Pixel Statistics Across CT Slices', color='white', fontsize=13, fontweight='bold')

for ax in axes:
    ax.set_facecolor(PANEL)
    ax.tick_params(colors=LABEL)
    for s in ax.spines.values(): s.set_edgecolor(BORDER)

# Left: mean HU distribution
for mod, col in [('CT', BLUE), ('CTA', GREEN)]:
    sub = stats_df[stats_df.modality == mod]
    axes[0].hist(sub.mean_hu, bins=20, color=col, alpha=0.7, edgecolor=BG, label=mod)
axes[0].set_title('Mean HU per Slice by Modality', color=TITLE)
axes[0].set_xlabel('Mean HU', color=LABEL); axes[0].set_ylabel('Count', color=LABEL)
axes[0].legend(labelcolor='white', facecolor='#21262d', edgecolor=BORDER)

# Right: mean HU vs std HU scatter
for mod, col in [('CT', BLUE), ('CTA', GREEN)]:
    sub = stats_df[stats_df.modality == mod]
    axes[1].scatter(sub.mean_hu, sub.std_hu, c=col, alpha=0.75, s=40, label=mod, edgecolors='none')
axes[1].set_title('Mean HU vs Std HU per Slice', color=TITLE)
axes[1].set_xlabel('Mean HU', color=LABEL); axes[1].set_ylabel('Std HU', color=LABEL)
axes[1].legend(labelcolor='white', facecolor='#21262d', edgecolor=BORDER)

plt.tight_layout()
plt.show()

---
## 1️⃣2️⃣  File Size: DICOM vs PNG vs JPEG

In [ ]:
brain_pil = Image.fromarray(apply_window(hu, 40, 80))

qualities = [10, 25, 40, 60, 75, 90, 95]
jpeg_kb = []
for q in qualities:
    buf = io.BytesIO()
    brain_pil.save(buf, format='JPEG', quality=q)
    jpeg_kb.append(buf.tell() / 1024)

png_buf = io.BytesIO()
brain_pil.save(png_buf, format='PNG')
png_kb = png_buf.tell() / 1024
dicom_kb = hu.nbytes / 1024

fig, ax = plt.subplots(figsize=(10, 5))
fig.patch.set_facecolor(BG)
ax.set_facecolor(PANEL)

ax.plot(qualities, jpeg_kb, 'o-', color=PINK, lw=2, markersize=7, label='JPEG (8-bit, lossy)')
ax.axhline(png_kb,   color=GREEN, lw=2, ls='--', label=f'PNG (8-bit, lossless) {png_kb:.0f} KB')
ax.axhline(dicom_kb, color=BLUE,  lw=2, ls='-.', label=f'DICOM raw pixels (16-bit) {dicom_kb:.0f} KB')

ax.set_xlabel('JPEG Quality', color=LABEL); ax.set_ylabel('File Size (KB)', color=LABEL)
ax.set_title('File Size vs Quality — why JPEG is banned from clinical DICOM workflows',
             color='white', fontsize=11)
ax.tick_params(colors=LABEL)
ax.legend(labelcolor='white', facecolor='#21262d', edgecolor=BORDER, fontsize=10)
for s in ax.spines.values(): s.set_edgecolor(BORDER)

plt.tight_layout()
plt.show()

---
## 1️⃣3️⃣  Export Utility — DICOM → PNG/NPY

In [ ]:
def batch_export(file_list, out_dir,
                 fmt='png', wc=40, ww=400,
                 max_files=None):
    """
    Export DICOM files from the kmader/siim-medical-images dataset.

    Parameters
    ----------
    file_list : list of .dcm paths
    out_dir   : output directory
    fmt       : 'png' for display-ready images,
                'npy' for raw HU arrays (no precision loss)
    wc, ww    : window centre / width used when fmt='png'
    max_files : limit number of files processed
    """
    os.makedirs(out_dir, exist_ok=True)
    if max_files:
        file_list = file_list[:max_files]

    for path in file_list:
        stem = os.path.splitext(os.path.basename(path))[0]
        try:
            d = load_dcm(path)
            h = to_hu(d)
            if fmt == 'png':
                out = os.path.join(out_dir, stem + '.png')
                Image.fromarray(apply_window(h, wc, ww)).save(out)
            elif fmt == 'npy':
                out = os.path.join(out_dir, stem + '.npy')
                np.save(out, h.astype(np.float32))
        except Exception as e:
            print(f'  ✗ {stem}: {e}')

    saved = len(glob.glob(os.path.join(out_dir, f'*.{fmt}')))
    print(f'✓ Exported {saved} {fmt.upper()} files → {out_dir}')


# Example usage (uncomment to run):
# batch_export(dcm_files, './exports_png', fmt='png', wc=40, ww=400)
# batch_export(dcm_files, './exports_npy', fmt='npy')

print('batch_export() helper ready.')
print('NOTE: Use fmt="npy" to preserve 16-bit HU precision for ML pipelines.')
print('      Use fmt="png" only for visualisation — diagnostic precision is lost.')

---
## ✅  Summary

| Step | What was done |
|---|---|
| Dataset | `kmader/siim-medical-images` — 475 CT series, 69 patients |
| Metadata EDA | Age distribution, CT vs CTA split, contrast usage |
| HU conversion | `raw × RescaleSlope + RescaleIntercept` |
| Windowing | Brain / Subdural / Stroke / Bone / Lung / Abdomen presets |
| Dashboard | Edge map, HU profile, tissue segmentation, stats panel |
| Batch stats | Mean & std HU across all slices, CT vs CTA comparison |
| Export | PNG (display, 8-bit) or NPY (full HU, lossless, ML-ready) |

### 🔗 References
- [gpreda's original notebook](https://www.kaggle.com/code/gpreda/visualize-ct-dicom-data/notebook)
- [kmader/siim-medical-images dataset](https://www.kaggle.com/datasets/kmader/siim-medical-images)
- [pydicom documentation](https://pydicom.github.io/pydicom/stable/)
- [DICOM standard](https://www.dicomstandard.org/)